# HW2 – Final pipeline 1. Data strategy tests

Four changes to the training data of RT-DETR-L, the best detector so far (notebook 05). Each is trained alone under the same protocol and budget as `rtdetr_lr` and compared with it:

1. **Masked ignore regions** – ignore regions can hold unlabeled objects, which the model learns as background; they are painted grey in the train frames.
2. **Low-resolution simulation** – 16% of the test frames are 680 or 960 px wide, a size absent from train and val; 20% of the train frames are shrunk to these sizes and enlarged back.
3. **Less mosaic** – mosaic on half of the images instead of all, so the model also sees normal frames.
4. **Tile-based training** – 960×960 crops of frames with a 1920 px long side, so that small objects are not shrunk. This is a separate model, meant for the ensemble.

The first three are candidates for the long RT-DETR runs; the fourth is a separate model for the ensemble.

## 0. Setup

In [1]:
# Make the repo importable: Kaggle clones it, locally the root is two levels above notebooks/final_pipeline/
import os, sys
from pathlib import Path
if os.path.exists("/kaggle/working"):
    if not os.path.exists("/kaggle/working/repo"):
        !git clone --depth 1 https://github.com/BohdanZasymovych/ai-hw2-object-detection.git /kaggle/working/repo
    !pip install -q pycocotools huggingface_hub kaggle ultralytics==8.4.207   # only if missing; torch/torchvision stay Kaggle's own CUDA builds
    ROOT_DIR = Path("/kaggle/working/repo")
else:
    ROOT_DIR = Path.cwd().parents[1] if Path.cwd().parent.name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT_DIR))

In [2]:
# Live reload of utils/common.py when editing locally (Kaggle's IPython autoreload is broken on Python 3.13)
if not os.path.exists("/kaggle/working"):
    %load_ext autoreload
    %autoreload 2
import json
import pandas as pd
import torch
import ultralytics
from utils.common import *
from utils.ultra import ULTRA_VERSION, YOLO_DATA_DIR, export_yolo, load_ultra, train_runs, ultra_predictor

seed_everything()
print(env_info(), "| ultralytics:", ultralytics.__version__, "| data:", RAW_DIR)
if ultralytics.__version__ != ULTRA_VERSION:
    print(f"Warning: tested with ultralytics {ULTRA_VERSION}; the patches in utils/ultra.py depend on its internals")

{'python': '3.14.4', 'torch': '2.15.0+cu132', 'torchvision': '0.29.2+cu132', 'gpu': 'NVIDIA GeForce RTX 2050', 'on_kaggle': False} | ultralytics: 8.4.207 | data: /home/bohdan/code/ai/hw2-object-detection/data/raw


Stages run in this session are listed in `RUN`; finished stages are downloaded from the Hugging Face repo. On Kaggle with **GPU T4 ×2** a Run All takes about 3 h: each round trains two runs concurrently, one per GPU. Locally nothing is trained and the tables are rebuilt from the saved runs. `QUICK=True` is a local smoke test on a few frames.

In [4]:
QUICK = False                                    # True: local smoke test (few frames, 1 epoch, small input, nothing pushed)
ALL_STAGES = {"ignore", "lowres", "mosaic", "tile", "reference"}
RUN = ALL_STAGES if os.path.exists("/kaggle/working") or QUICK else set()   # stages computed in this session; the rest are pulled from the HF repo

# Fail early if trained stages could not be uploaded (needs HF_TOKEN with write access)
if RUN and not QUICK:
    check_hf_write()

## 1. Training setup

Same as `rtdetr_lr` in notebook 05: RT-DETR-L with COCO weights, 960 px input, AdamW, lr 1e-4, backbone lr ×0.1, 100 denoising queries, batch 4, nominal batch 16, 1.1 h of training per run. The same val frames and the shared `coco_evaluate` are used throughout.

The tile frames have a 1920 px long side, so all test frames share one tile layout (6 windows of 960 px). Each frame gives one crop during training, so an epoch has the same number of images as the other runs.

In [5]:
CFG = dict(
    arch="rtdetr-l", weights="rtdetr-l.pt", long_side=640 if QUICK else PREP["long_side"],
    train_frames=16 if QUICK else None, val_frames=8 if QUICK else None, num_workers=4, seed=SEED,
    train=dict(  # Ultralytics train() arguments of rtdetr_lr (notebook 05)
        imgsz=320 if QUICK else 960, batch=2 if QUICK else 4, epochs=1 if QUICK else 100, time=None if QUICK else 1.1,
        optimizer="AdamW", lr0=1e-4, weight_decay=1e-4, nbs=16, warmup_epochs=0 if QUICK else 0.5, warmup_bias_lr=0.0,
        close_mosaic=0 if QUICK else 1, scale=0.25, workers=2 if QUICK else 4, seed=SEED, deterministic=False,
        max_det=EVAL_DETS, plots=False, cache=False),
)
MODEL = dict(backbone_lr_ratio=0.1, nd=100)
TEST_IMGSZ = 640 if QUICK else 1280                 # test input of rtdetr_final, the model to beat
TILE = 320 if QUICK else 960                        # tile size, input of the tile model
TILE_SIDE = 640 if QUICK else 1920                  # long side of the frames that are cut into tiles
LOW_SIDES = (480, 320) if QUICK else (960, 680)     # low resolutions, as in the test frames

STAGES = {   # export_yolo options and Ultralytics train() overrides of each test
    "ignore": dict(export=dict(mask_ignore=True)),
    "lowres": dict(export=dict(lowres={s: 0.1 for s in LOW_SIDES})),
    "mosaic": dict(train=dict(mosaic=0.5)),
    "tile": dict(export=dict(long_side=TILE_SIDE, tile=TILE)),
}


def name(stage):
    return f"s1_{stage}" + ("_quick" if QUICK else "")

## 2. Data export and helpers

In [6]:
def dataset(stage):
    # Data yaml of a test; a test without export options uses the standard dataset of notebook 05
    export = STAGES[stage].get("export", {})
    folder = (f"quick_{stage}" if QUICK else stage) if export else ("quick" if QUICK else str(CFG["long_side"]))
    return export_yolo(YOLO_DATA_DIR / folder, **{"long_side": CFG["long_side"], "max_frames": CFG["train_frames"], **export})


def ensure_run(run, weights=False):
    # Local folder of a finished run; downloaded from the Hugging Face repo if it was produced in another session
    return fetch_run(run, weights, hint=f"Add its stage to RUN to compute it in this session (RUN = {RUN}).")


def predict_val(model, tile=False, degrade_to=None):
    # Val predictions: the whole frame at the test input of rtdetr_final, or all tiles of the 1920 px frame
    side = TILE_SIDE if tile else TEST_IMGSZ
    predictor = ultra_predictor(model, TILE if tile else TEST_IMGSZ, nms=0.6 if tile else 0.7, tile=TILE if tile else False)
    ds = DetDataset("val", label_offset=0, max_frames=CFG["val_frames"], long_side=side, degrade_to=degrade_to)
    preds, ms = predict_dataset(predictor, ds, batch_size=1, num_workers=CFG["num_workers"])
    return preds, ds.image_ids, ms


def low_res_map(model):
    # mAP on val frames shrunk to the low resolutions of the test set and enlarged back
    return {f"mAP at {s} px": coco_evaluate(predict_val(model, degrade_to=s)[0], "val", image_ids=predict_ids)["mAP"]
            for s in LOW_SIDES}


predict_ids = DetDataset("val", max_frames=CFG["val_frames"]).image_ids

In [7]:
def train_stages(stages):
    # Train the given tests under the fixed protocol, concurrently on 2 GPUs if available, then evaluate each best
    # epoch with the shared COCO evaluation, save and upload the run
    cfgs = {s: dict(name=name(s), data=str(dataset(s)), arch=CFG["arch"], weights=CFG["weights"], **MODEL,
                    train={**CFG["train"], **STAGES[s].get("train", {})}) for s in stages}
    train_runs(list(cfgs.values()))
    for s, c in cfgs.items():
        model = load_ultra(run_dir(c["name"]) / "best.pt")
        preds, ids, ms = predict_val(model, tile=s == "tile")
        metrics = {**coco_evaluate(preds, "val", image_ids=ids), "ms_per_img": ms}
        if s == "lowres":
            metrics.update(low_res_map(model))
        save_run(c["name"], config={**CFG, **STAGES[s], "model": MODEL, "image_ids": ids}, metrics=metrics, preds=preds)
        if not QUICK:
            push_run(c["name"])
        del model
        torch.cuda.empty_cache()

## 3. Round 1 – Masked ignore regions and low-resolution simulation

In [19]:
train_stages([s for s in ("ignore", "lowres") if s in RUN])

## 4. Round 2 – Less mosaic and tile-based training

In [20]:
train_stages([s for s in ("mosaic", "tile") if s in RUN])

## 5. Reference: the current model on tiles and low-resolution frames

`rtdetr_lr` needs no retraining here. Tile inference on it shows what tiling gives without tile training, and its low-resolution results are the baseline for the low-resolution test.

In [21]:
if "reference" in RUN:
    model = load_ultra(ensure_run("rtdetr_lr", weights=True) / "best.pt")
    preds, ids, ms = predict_val(model, tile=True)
    metrics = {**coco_evaluate(preds, "val", image_ids=ids), "ms_per_img": ms, **low_res_map(model)}
    save_run(name("reference"), config=dict(source="rtdetr_lr", image_ids=ids), metrics=metrics, preds=preds)
    if not QUICK:
        push_run(name("reference"))
    del model
    torch.cuda.empty_cache()

## 6. Comparison

Every row is compared with `rtdetr_final` (`rtdetr_lr` tested at 1280 px, mAP 0.282). **Test-like mAP** leaves out the val frames wider than 1920 px, which do not occur in the test set. **Videos better** counts the val videos with a higher mAP than `rtdetr_final`.

In [22]:
frames = subsample(load_split("val")[0])
frames = frames[frames.image_id.isin(predict_ids)]
test_like = frames[frames.width <= 1920].image_id.tolist()
videos = {seq: g.image_id.tolist() for seq, g in frames.groupby("seq")}


def video_map(preds):
    return {seq: coco_evaluate(preds, "val", image_ids=ids)["mAP"] for seq, ids in videos.items()}


def row(preds):
    m = coco_evaluate(preds, "val", image_ids=predict_ids)
    better = sum(a > b for a, b in zip(video_map(preds).values(), base_videos.values()))
    return {"mAP": m["mAP"], "AP_small": m["AP_small"], "test-like mAP": coco_evaluate(preds, "val", image_ids=test_like)["mAP"],
            "videos better": f"{better} / {len(videos)}"}


load_preds = lambda run: json.load(open(ensure_run(run) / "preds_val.json"))
base, yolo, tile = load_preds("rtdetr_final"), load_preds("yolo_final"), load_preds(name("tile"))
base_videos = video_map(base)
table = pd.DataFrame({
    "rtdetr_final (baseline)": row(base),
    "masked ignore regions": row(load_preds(name("ignore"))),
    "low-resolution simulation": row(load_preds(name("lowres"))),
    "less mosaic": row(load_preds(name("mosaic"))),
    "rtdetr_lr, tile inference": row(load_preds(name("reference"))),
    "tile model, tile inference": row(tile),
    "WBF: rtdetr_final + YOLO11s": row(wbf([base, yolo])),
    "WBF: rtdetr_final + tile model": row(wbf([base, tile])),
}).T
table.insert(1, "Δ mAP vs baseline", table["mAP"] - table.loc["rtdetr_final (baseline)", "mAP"])
table.sort_values("Δ mAP vs baseline", ascending=False).round(3)

,mAP,Δ mAP vs baseline,AP_small,test-like mAP,videos better
WBF: rtdetr_final + tile model,0.311219,0.029316,0.148917,0.324456,7 / 7
"tile model, tile inference",0.310086,0.028183,0.154383,0.32516,7 / 7
WBF: rtdetr_final + YOLO11s,0.298438,0.016535,0.139324,0.315689,6 / 7
less mosaic,0.289977,0.008075,0.130605,0.305038,7 / 7
low-resolution simulation,0.2858,0.003897,0.13725,0.299939,4 / 7
rtdetr_final (baseline),0.281903,0.0,0.137304,0.300046,0 / 7
"rtdetr_lr, tile inference",0.279994,-0.001909,0.15485,0.293889,5 / 7
masked ignore regions,0.275146,-0.006756,0.136374,0.295798,3 / 7


Normal val has no low-resolution frames, so the low-resolution test is also scored on val frames shrunk to these sizes and enlarged back.

In [23]:
low = lambda run: {k: v for k, v in json.load(open(ensure_run(run) / "metrics.json")).items() if k.startswith("mAP at")}
table = pd.DataFrame({"rtdetr_lr (baseline)": low(name("reference")), "low-resolution simulation": low(name("lowres"))}).T
for col in list(table.columns):
    table[f"Δ {col}"] = table[col] - table.loc["rtdetr_lr (baseline)", col]
table["mean Δ"] = table[[c for c in table.columns if c.startswith("Δ")]].mean(axis=1)
table.sort_values("mean Δ", ascending=False).round(3)

,mAP at 960 px,mAP at 680 px,Δ mAP at 960 px,Δ mAP at 680 px,mean Δ
low-resolution simulation,0.279,0.247,0.012,0.017,0.014
rtdetr_lr (baseline),0.267,0.230,0.000,0.000,0.000


## 7. Public leaderboard

An optional second check, run after the tests have finished. Val has a 4K video that the test set lacks and none of its low-resolution frames, so the public score measures something val cannot. `ignore`, `lowres` and `mosaic` were submitted already (scores below). The tile model is submitted at 960 px tiles of the 1920 px frame, with the same tile inference as on val.

Set `SUBMIT = True` and run the notebook locally (nothing is trained with `RUN` empty). Every submission uses one of the daily submissions, so set it back to `False` afterwards. An existing CSV is reused, so a run interrupted between models continues with the next one.

In [8]:
SUBMIT = True                                  # True: predict the test frames of the stages below and submit them to Kaggle
SUBMIT_STAGES = ["tile"]                        # submitted now; ignore, lowres and mosaic were submitted already
PUBLIC = {"rtdetr_final (baseline, notebook 05)": 0.2590, "ignore": 0.2693, "lowres": 0.2661, "mosaic": 0.2624}   # public scores so far
TEST = {s: dict(imgsz=TEST_IMGSZ, long_side=TEST_IMGSZ) for s in ("ignore", "lowres", "mosaic")}
TEST["tile"] = dict(imgsz=TILE, long_side=TILE_SIDE, nms=0.6, tile=TILE)    # six overlapping 960 px tiles of the 1920 px frame

if SUBMIT and not QUICK:
    public = dict(PUBLIC)
    for s in SUBMIT_STAGES:
        path = SUBMISSIONS_DIR / f"submission_{name(s)}.csv"
        t = TEST[s]
        if not path.exists():
            model = load_ultra(ensure_run(name(s), weights=True) / "best.pt")
            make_submission(ultra_predictor(model, t["imgsz"], nms=t.get("nms", 0.7), tile=t.get("tile", False)), name(s),
                            long_side=t["long_side"], label_offset=0, num_workers=CFG["num_workers"])
            del model
            torch.cuda.empty_cache()
        row = submit_to_kaggle(path, f"{name(s)}: RT-DETR-L, " + (f"{TILE} px tiles of {TILE_SIDE} px frames" if "tile" in t else f"{t['imgsz']} px"))
        public[s] = float(row["publicScore"]) if row else None
    table = pd.Series(public, name="public score").astype(float).to_frame()
    table["Δ vs baseline"] = table["public score"] - table.loc["rtdetr_final (baseline, notebook 05)", "public score"]
    display(table.sort_values("Δ vs baseline", ascending=False).round(4))

/home/bohdan/code/ai/hw2-object-detection/submissions/submission_s1_tile.csv: 3811 frames, 1143300 boxes, 391 ms/img, 36 MB
Submitted /home/bohdan/code/ai/hw2-object-detection/submissions/submission_s1_tile.csv: s1_tile: RT-DETR-L, 960 px tiles of 1920 px frames
status: SubmissionStatus.COMPLETE | public score: 0.2628873158


## 8. Conclusion

Four changes were tested against `rtdetr_final` (val mAP 0.282, public 0.2590). The tile model is the largest gain; the three data changes are small but carried into session 2.

- **Tile-based training** (val 0.310, +0.028, all 7 videos better). It is a separate ensemble member. Its public score is still to be checked.
- **Masked ignore regions** (val 0.275, −0.007; public 0.2693, +0.010). The val loss comes almost entirely from the bus class (264 val boxes). It is kept, since the public gain is larger.
- **Low-resolution simulation** (public 0.2661, +0.007). It gains 0.012–0.017 mAP on degraded val and leaves clean val unchanged. It targets the 16% of test frames at 680–960 px.
- **Mosaic at 0.5** (val 0.290, +0.008, all 7 videos better, most by a small margin; public 0.2624, +0.003). Vehicle classes gain, small objects lose slightly.

**Ensemble:** WBF of the tile model, mosaic and YOLO11s reaches 0.328 on val, against 0.282 for the baseline. This is the best of six combinations on the same seven videos, so it is optimistic.

**Carried into session 2:** masked ignore regions, low-resolution simulation and mosaic at 0.5. Tile-based training goes into session 2 or 3 as a separate model, depending on its public score.